# Obtaining systemic redshifts from stacked emission

This script estimates the systemic redshift of each source by stacking optically-thin emission lines and fitting gaussian profiles

In [ ]:
import numpy as np
from astropy.io import fits
import astropy.table as aptb
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import copy

from tangelo import spectroscopy as tgspec
from tangelo import models as tgmod
from tangelo import fitting as tgfit
from tangelo import catalogue_operations as tgcat
from tangelo import constants as tgconst
from tangelo import io as tgio

In [ ]:
# Load the megatab
SPEC_SOURCE = "R21"  #  Where are the spectra from? 'R21' for Richard+21, 'APER' for apertures
SPEC_TYPE   = "weight_skysub" # Which subtype of spectrum to use (e.g. 'Nfwhm_opt' for APER, 'weight_skysub' for R21)   

tabfile = f"megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_{SPEC_TYPE}.fits"
tabhdul = fits.open(tabfile)
megatable = aptb.Table(tabhdul[1].data)

In [ ]:
# Get the dictionary of lines to look for
wavedict = tgconst.wavedict

# Clean the megatable to only include spectra with good SNR in red or blue Lya peak
megatab = megatable[(megatable['SNRR'] > 5.0) + (megatable['SNRB'] > 5.0)]

In [ ]:
# First try to get as many measurements of sys vel as possible
# Try to re-do the Z_sys calculation for all sources, this time by averaging all the optically thin lines
# (NOT CIV) and fitting gaussians

# Fitting configuration dictionary
fit_config = {
    'sig_level'   : 3, # What significance level to accept a fit at
    'search_width': 600, # Half width of the region around the Lyman alpha peak to search for stacked emission
    'fit_width'   : 1500, # Half width of the fitting region around Lya peak
    'vel_bin'     : 50
}



# These are the lines that, if already detected at >3 sigma, can be used to make the stack
good_lines =  ['CIII1907', 'CIII1909', 'HeII1640', 
               'NIII1750', 'NIV1483', 'NIV1487',
               'OIII1660', 'OIII1666', 'SiIII1883', 'SiIII1892']

# These are the default optically thin lines to use if no good lines are detected
optthin_lines = ['HeII1640','OIII1666','CIII1907','CIII1909']

megatab_newsysz = copy.deepcopy(megatab) # make a copy to add new columns to
# Add new columns to table
megatab_newsysz.add_columns([aptb.Column(np.nan*np.zeros(len(megatab)), "DELTAV_LYA"),
                            aptb.Column(np.nan*np.zeros(len(megatab)), "DELTAV_LYA_ERR"),
                            aptb.Column(np.nan*np.zeros(len(megatab)), "VFWHM_SYS"),
                            aptb.Column(np.nan*np.zeros(len(megatab)), "VFLUX_SYS"),
                             aptb.Column(np.nan*np.zeros(len(megatab)), "VFLUX_SYS_ERR"),
                            aptb.Column(np.nan*np.zeros(len(megatab)), "VCONT_SYS"),
                            aptb.Column(np.nan*np.zeros(len(megatab)), "VSLOPE_SYS")])

hi_abslines = ['SiIV1394','SiIV1403'] # Abs lines to be plotted to compare with the sys z
li_abslines =  ['SiII1260','CII1334','SiII1527'] # Abs lines to be plotted to compare with the sys z

det_lines = [] # Initialise list of detected lines across all sources
count = 0 # Initialise count of significant zsys measurements
emitters = 0 # Initialise count of statistically significant emitters
for i, row in enumerate(megatab_newsysz):
    cluster = row['CLUSTER']
    iden = row['iden']
    print("\n" + "=" * 80)
    print(f"\nProcessing {iden} from {cluster}...")
    # Check for tentative unflagged emission lines found in previous step
    tentem = tgcat.is_true_emitter(row, {x: wavedict[x] for x in good_lines}, n=2, sig=3, return_lines=True)
    if tentem[0]:
        print('Verified emitter, using all detected lines at 3 sigma:')
        emlines = list(
            set(tentem[1][0])
            & set(good_lines)
        )
        print(emlines)
        emitters += len(emlines) > 0
        det_lines = det_lines + emlines
        if len(emlines) == 0:
            print('Insufficient optically thin lines. Reverting to default line list...')
            emlines = optthin_lines
    else:
        print(f"Insufficient emission lines found. Using default line list.")
        emlines = optthin_lines
    
    # Get the average line profile as well as Lyman alpha and absorption lines for comparison
    z_lya = row['LPEAKR'] / 1215.67 - 1.
    velbounds = [-2500, 2500]
    vel, spec, specerr = tgspec.avg_lines(row, emlines, absorption=False, velbounds=velbounds, z = z_lya, 
                                          spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE, velstep=fit_config['vel_bin'])
    lyvel, lyspec, lyspecerr = tgspec.avg_lines(row, ['LYALPHA'], absorption=False, velbounds=velbounds, z = z_lya, 
                                                spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE, velstep=fit_config['vel_bin'])
    abvel, abspec, abspecerr = tgspec.avg_lines(row, li_abslines, absorption=False, velbounds=velbounds, z = z_lya, 
                                                spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE, velstep=fit_config['vel_bin'])
    habvel, habspec, habspecerr = tgspec.avg_lines(row, hi_abslines, absorption=False, velbounds=velbounds, z = z_lya, 
                                                   spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE, velstep=fit_config['vel_bin'])
    
    # Check for NaNs in the averaged line profile, which will indicate no lines were found
    if np.sum(np.isnan(vel * spec * specerr)) > 0:
        print("Source redshift is too high, no lines to fit. Skipping.\n")
        continue

    # Plot the line profiles for visual inspection
    fig, ax = plt.subplots(4, 1, figsize=(5,5), facecolor='white', sharex=True)
    fig.subplots_adjust(hspace=0)
    ax[0].set_title(f"{row['CLUSTER']} ID {row['iden']}: $ z = {row['z']:.4f} $")
    ax[1].set_xlabel(r'$v$ [km\,s$^{-1}$]')
    ax[1].set_ylabel(r'$f_v$ [$10^{-20}$\,erg\,s$^{-1}$\,cm$^{-2}$\,(km\,s$^{-1}$)$^{-1}$]')
    ax[0].plot(lyvel, lyspec, drawstyle='steps-mid', color='tab:red')
    ax[0].fill_between(lyvel, lyspec - lyspecerr, lyspec + lyspecerr, step='mid', facecolor='tab:red', alpha=0.3, linestyle='')
    ax[1].plot(vel, spec, drawstyle='steps-mid', color='tab:orange')
    ax[1].fill_between(vel, spec - specerr, spec + specerr, step='mid', facecolor='tab:orange', alpha=0.3, linestyle='')
    ax[2].plot(abvel, abspec, drawstyle='steps-mid', color='tab:blue')
    ax[2].fill_between(abvel, abspec - abspecerr, abspec + abspecerr, step='mid', facecolor='tab:blue', alpha=0.3, linestyle='')
    ax[3].plot(habvel, habspec, drawstyle='steps-mid', color='tab:purple')
    ax[3].fill_between(habvel, habspec - habspecerr, habspec + habspecerr, step='mid', facecolor='tab:purple', alpha=0.3, linestyle='')
    
    # Now fit a gaussian to the average line profile
    fitreg = (-fit_config['fit_width'] < vel) * (vel < fit_config['fit_width'])
    fitreg &= ~(spec * specerr == 0) # some spectra have zero errors which will cause infinite normalised residuals in curve_fit
    # Use a range of initial guesses to try to ensure the fitter finds the global minimum, not a local one
    search_reg = (-fit_config['search_width'] < vel) & (vel < fit_config['search_width'])
    max_idx = np.argmax(spec[search_reg])
    peak_vel = vel[search_reg][max_idx]
    vpeak_guesses = [peak_vel] # first guess is on the highest point within the search bounds
    vpeak_guesses.extend([-400., -200., 200., 400.]) # subsequent generic guesses
    initial_guess = {
        'FLUX': (np.nanmax(spec)-np.nanmedian(spec)) * np.min(np.ediff1d(vel)),
        'VPEAK': 0,
        'FWHM': 100,
        'CONT': np.nanmedian(spec),
        'SLOPE': 0
        }
    initial_guess_series = {vpeak : {**initial_guess, 'VPEAK': vpeak} for vpeak in vpeak_guesses}
    # Perform an initial fit to get starting parameters for MCMC and judge
    # whether it is worth proceeding with the MCMC fitting (which is more computationally expensive)
    print("Performing initial fits to judge significance of line detection...")
    initial_fits = {}
    best_vpeak = None
    attempts = 3
    for vpeak, guesses in initial_guess_series.items():
        print(f"Fitting with initial VPEAK = {vpeak:.2f}")
        popt, pcov = np.zeros(len(guesses)), np.ones((len(guesses), len(guesses)))
        for n in range(attempts):
            try:
                popt, pcov = curve_fit(tgmod.gaussian, vel[fitreg], spec[fitreg], sigma=specerr[fitreg],
                                        p0 = list(guesses.values()), absolute_sigma=True, max_nfev=100000,
                                        bounds = [[0, -fit_config['search_width'], 25, -50, -1],
                                                [1000, fit_config['search_width'], 200, 1000, 1]], method='trf')
                break
            except RuntimeError:
                print(f"Fitting attempt {n+1} failed due to Runtime Error.")
                if n+1 < attempts:
                    print(f"Trying again...")
                    continue
                else:
                    print(f"All fit attempts failed. Moving to next guess...")
                    pass
            
        initial_fits[vpeak] = (popt, pcov)
        # Compare the snr of the fit to previous fits to determine which is best
        snr = popt[0] / np.sqrt(pcov[0][0])
        if best_vpeak is None or snr > initial_fits[best_vpeak][0][0] / np.sqrt(initial_fits[best_vpeak][1][0][0]):
            best_vpeak = vpeak

    best_snr = initial_fits[best_vpeak][0][0] / np.sqrt(initial_fits[best_vpeak][1][0][0])
    if best_snr < 1:
        print("Initial fit < 2σ. Moving to next source.\n")
        plt.show()
        plt.close()
        continue
    else:
        print("Initial fit > 2σ, proceeding with MCMC fitting...")

    # Determine which initial guess gives the better fit and use that as the starting point for MCMC
    best_guesses = initial_fits[best_vpeak][0]
    guesses_dict = dict(zip(initial_guess.keys(), best_guesses))
    bounds_dict = {
        'FLUX': (0,1000),
        'VPEAK': (-fit_config['search_width'], fit_config['search_width']),
        'FWHM': (25, 200),
        'CONT': (-np.inf, np.inf),
        'SLOPE': (-np.inf, np.inf)
    }

    print(f"Fitting with VPEAK initial guess = {guesses_dict['VPEAK']}...")
    bootstrap_params = {
        'niter': 1000,
        'max_nfev': 10000,
        'autocorrelation': True,
        'errfunc': 'stddev_7',
        'baseline_order': 1
    }
    fit_result_dict = tgfit.fit_line(vel[fitreg], spec[fitreg], specerr[fitreg], "STACK",
                                     guesses_dict, bounds = bounds_dict, continuum_buffer=1500,
                                     plot_result=False, bootstrap_params=bootstrap_params)

    param_dict = fit_result_dict['param_dict']
    error_dict = fit_result_dict['error_dict']
    popt_mc = [param_dict[key] for key in initial_guess.keys()]
    perr_mc = [error_dict[key] for key in initial_guess.keys()]
    multipeak_flag = fit_result_dict['multipeak_flag']
    significant = popt_mc[0]/perr_mc[0] > fit_config['sig_level']
    # This condition is designed to catch cases where there are suspiciously high continuum levels
    # in a source that allegedly had no HST continuum detection (which suggests contamination rather than
    # genuinely high continuum)
    continuum_anomaly_flag = popt_mc[-2] > 0.5 and row['idfrom'] == 'MUSELET'

    bad_fit = any([multipeak_flag, continuum_anomaly_flag, not significant])
    fit_color = 'grey' if bad_fit else 'fuchsia'

    velhires = np.linspace(vel[0], vel[-1], 1000)
    ax[1].plot(velhires, tgmod.gaussian(velhires, *popt_mc), linestyle='--', color=fit_color, alpha=0.6,
                    label='model')
    for a in ax:
        a.axvline(popt_mc[1], linestyle='--', color='black', alpha=0.4)
    plot_dir = tgio.get_plot_dir(cluster, iden)
    plt.savefig(plot_dir / f"{cluster}_ID{iden}_sysz_fit.png", dpi=300)
    plt.show()
    plt.close()

    if not bad_fit:
        print(f"Found a significant line detection with VPEAK = {popt_mc[1]:.1f} km/s and FWHM = {popt_mc[2]:.1f} km/s.")
    elif multipeak_flag:
        print(f"Fit is flagged as multi-peaked, which may indicate a spurious fit. Moving to next source.\n")
    elif continuum_anomaly_flag:
        print(f"Fit is flagged as having an anomalously high continuum level, which may indicate contamination. Moving to next source.\n")
    else:
        print(f"Fit is not statistically significant at the {fit_config['sig_level']}σ level. Moving to next source.\n")
    
    count += int(significant)
    print(str(count)+'/'+str(i+1))

    if bad_fit:
        continue
    
    print(f"Significant emission found. Entering sys z info into table...")
    megatab_newsysz['DELTAV_LYA'][i] = -popt_mc[1]
    megatab_newsysz['DELTAV_LYA_ERR'][i] = np.abs(perr_mc[1])
    megatab_newsysz['VFWHM_SYS'][i] = popt_mc[2]
    megatab_newsysz['VFLUX_SYS'][i] = popt_mc[0]
    megatab_newsysz['VFLUX_SYS_ERR'][i] = perr_mc[0]
    megatab_newsysz['VCONT_SYS'][i] = popt_mc[-2]
    megatab_newsysz['VSLOPE_SYS'][i] = popt_mc[-1]
    print(f"Done.\n")

In [ ]:
# Save the new megatable with systemic redshift info
megatab_newsysz.write(f"./megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_{SPEC_TYPE}.fits", overwrite=True)

In [ ]:
# Check how many Lya velocity offsets are negative

N_neg = (megatab_newsysz['DELTAV_LYA'] < 0).sum()
N_pos = (megatab_newsysz['DELTAV_LYA'] > 0).sum()
f_neg = N_neg / (N_neg + N_pos)

print(f"Number (fraction) of negative Lyα velocity offsets:")
print(f"{N_neg} ({f_neg:.2f})")

In [ ]:
plt.scatter(megatab_newsysz['z'], megatab_newsysz['DELTAV_LYA'])